In [ ]:
#imports
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import random
from pathlib import Path
import os 
import sys
from collections import Counter

In [ ]:
#reload notebook
%load_ext autoreload
%autoreload 2

In [ ]:
#Directory set up
BASE_DIR = Path.cwd() 
print(BASE_DIR)

c:\Users\Neezy\Videos\attacks_improved_v2\Attacks-Improved_v2\Random_forest


In [ ]:
project_root = os.path.abspath('..')
if project_root not in sys.path: 
    sys.path.append(project_root)

In [5]:
#### log initialization
from utils.logs import logger_module 
logger = logger_module(logger_name=__name__) 

In [6]:
# import attacks
from attack_functionality.general_attack import GeneralAttack 
attack = GeneralAttack()

In [ ]:
#Read dataset
data = pd.read_csv(BASE_DIR.parent/'data/ds_with_sample_name.csv')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [ ]:
#model for selecting important features
X = data.iloc[:, 2:-1].values
y = data.iloc[:, -1].values
original_indices = np.arange(len(X)) 
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(X, y, original_indices, test_size=0.2, random_state=42)
classifier = RandomForestClassifier(n_estimators=100, random_state=42)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')
cm = confusion_matrix(y_test, y_pred)
print(f"Confusion Matrix:\n {cm}") 
cr = classification_report(y_test, y_pred)
print(f"Classification report:\n {cr}")

Accuracy: 97.60%
Confusion Matrix:
 [[104  13]
 [  1 465]]
Classification report:
               precision    recall  f1-score   support

           0       0.99      0.89      0.94       117
           1       0.97      1.00      0.99       466

    accuracy                           0.98       583
   macro avg       0.98      0.94      0.96       583
weighted avg       0.98      0.98      0.98       583



In [ ]:
#Identify important features based on feature_importances
importances = classifier.feature_importances_
feature_imp_df = pd.DataFrame({'Feature': data.iloc[:, 2:-1].columns, 'Gini Importance': importances}).sort_values(
    'Gini Importance', ascending=False)

important_features = []
importances_to_numpy = feature_imp_df.to_numpy()
for i in range(0,20):
     important_features.append(importances_to_numpy[i][0])

In [10]:
important_features.append('label')

In [11]:
df = data[important_features]

In [12]:
df.head()

,DrawTextExW,RegEnumKeyExW,RegEnumKeyW,SetFileTime,FindResourceExW,CreateDirectoryW,GetDiskFreeSpaceExW,RegQueryValueExW,GetFileInformationByHandleEx,LoadResource,FindWindowExA,RemoveDirectoryW,SearchPathW,RegOpenKeyExW,NtReadFile,NtOpenMutant,GetSystemTimeAsFileTime,GetSystemDirectoryW,CoUninitialize,GetForegroundWindow,label
0,86,128,8,3317,32,9,0,1308,3,31,0,0,0,1547,11208,1,192,25,40,4,1
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1
2,0,38,0,0,1,3,0,317,0,0,0,0,276,195,459,2,39,11,0,0,1
3,0,21,0,0,0,4,0,182,0,0,0,0,0,106,1118,2,102,4,0,0,1
4,0,0,0,0,0,0,0,0,0,2,0,0,0,0,1,0,5,1,1,0,1


In [ ]:
#Train surrogate model 
X_imp_feat = df.iloc[:, :-1].values
y_imp_feat = df.iloc[:, -1].values
original_indices = np.arange(len(X_imp_feat)) 
X_train_fi, X_test_fi, y_train_fi, y_test_fi, idx_train, idx_test = train_test_split(X_imp_feat, y_imp_feat, original_indices, test_size=0.2, random_state=42)
classifier_rf_imp = RandomForestClassifier(n_estimators=100, random_state=42)
classifier_rf_imp.fit(X_train_fi, y_train_fi)
y_pred_imp = classifier_rf_imp.predict(X_test_fi)
accuracy_fi = accuracy_score(y_test_fi, y_pred_imp)
print(f'Accuracy: {accuracy_fi * 100:.2f}%')
cm_fi = confusion_matrix(y_test_fi, y_pred_imp)
print(f"Confusion Matrix:\n {cm_fi}") 
cr_fi = classification_report(y_test_fi, y_pred_imp)
print(f"Classification report:\n {cr_fi}")

Accuracy: 97.43%
Confusion Matrix:
 [[104  13]
 [  2 464]]
Classification report:
               precision    recall  f1-score   support

           0       0.98      0.89      0.93       117
           1       0.97      1.00      0.98       466

    accuracy                           0.97       583
   macro avg       0.98      0.94      0.96       583
weighted avg       0.97      0.97      0.97       583



In [14]:
# ransomware_training_sample_indices = np.where(y_train==1)
# idx = np.random.choice(ransomware_training_sample_indices[0], 580, replace=False)
# xtrain_poison_candidates = X_train_fi[idx] 


In [ ]:
idx_loaded = np.load(BASE_DIR.parent/"Adversarial_candidates_indices/dt_adversarial_candidates_indices.npy")
xtrain_poison_candidates = X_train_fi[idx_loaded] 
print(len(xtrain_poison_candidates))

580


In [ ]:
#define perturbation value
perturbation=80
#Generate adversarial samples
asamples, adv_idx = attack.generate_adversarial_sample(idx_loaded, xtrain_poison_candidates, perturbation, classifier_rf_imp)

The average  l2 distance 111.2997288752921 across 473 samples
The biggest  l2 distance 147.8817094843037 across 473 samples
The smallest l2 distance 63.30876716537765 across 473 samples
The average amount of features changed: 9.274841437632135
The biggest amount of features changed: 12
The least amount of features changed: 5


In [17]:
print(len(asamples))
print(f"AGR: {len(asamples)/len(xtrain_poison_candidates):.4f}")

473
AGR: 0.8155


In [19]:
### Adversarial Retrain ###  
mask = np.ones(X_train_fi.shape[0], dtype=bool)
mask[adv_idx] = False

X_train_fi = X_train_fi[mask]
y_train_fi = y_train_fi[mask]

print(f"length of Xtrain with successful base adversarial samples removed: {len(X_train_fi)}")

### Select a hold out test set of the generated adversarial samples
k = round(0.2 * len(asamples)) 
print(f"20% of the adversarial samples held out for testing: {k}")
remove_samples_from_asamples = random.sample(asamples, k)
print(len(remove_samples_from_asamples))

### remove test set from total adversarial samples and add adversarial samples to the training set 
result = [item for item in asamples if item not in remove_samples_from_asamples]
np_result = np.array(result)
print(len(result))
poisoned_xtrain = np.concatenate((X_train_fi, np_result))
print(f"Total length of poisoned Xtrain: {len(poisoned_xtrain)}")

### Add the corresponding amount of ytrain adversarial samples. (correctly labelled as ransomware) 
adv_y = []
for i in range(len(np_result)): 
    adv_y.append(1) 
    
print(len(adv_y))

np_advy = np.array(adv_y)
poisoned_ytrain = np.concatenate((y_train_fi, np_advy))
print(f"Total length of poisoned Ytrain: {len(poisoned_ytrain)}") 

if len(poisoned_ytrain) == len(poisoned_xtrain): 
    print("Poisoned Xtrain and ytrain are aligned, posioned training can now commence")
else: 
    print("Not aligned, something is not aligned, check k value")

length of Xtrain with successful base adversarial samples removed: 1857
20% of the adversarial samples held out for testing: 95
95
101
Total length of poisoned Xtrain: 1958
101
Total length of poisoned Ytrain: 1958
Poisoned Xtrain and ytrain are aligned, posioned training can now commence


In [ ]:
#Train Surrogate model with adversarial samples added to training set
poisoned_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
poisoned_classifier.fit(poisoned_xtrain, poisoned_ytrain)
y_pred_poisoned = poisoned_classifier.predict(X_test_fi)
poisoned_accuracy = accuracy_score(y_test_fi, y_pred_poisoned)
print(f'Accuracy trained on poisoned samples: {poisoned_accuracy * 100:.2f}%')
cm_poisoned = confusion_matrix(y_test_fi, y_pred_poisoned)
print(f"Confusion Matrix trained on poisoned samples:\n {cm_poisoned}") 
cr_poisoned = classification_report(y_test_fi, y_pred_poisoned)
print(f"Classification report trained on poisoned samples:\n {cr_poisoned}")

Accuracy trained on poisoned samples: 97.77%
Confusion Matrix trained on poisoned samples:
 [[105  12]
 [  1 465]]
Classification report trained on poisoned samples:
               precision    recall  f1-score   support

           0       0.99      0.90      0.94       117
           1       0.97      1.00      0.99       466

    accuracy                           0.98       583
   macro avg       0.98      0.95      0.96       583
weighted avg       0.98      0.98      0.98       583



In [ ]:
#Evaluate retrained surrogate model 
def make_pred_poisoned(samples):
    predictions = [] 
    for sample in samples:
        single_pred = poisoned_classifier.predict([sample])
        predictions.append(single_pred)
        print(single_pred) 
        if single_pred[0] == 1: 
            print("Correctly identified adversarial ransomware sample") 
        else: 
            print("Adversarial sample still adversarial")

    zero_counter = 0 
    one_counter = 0 
    for pred in predictions:
        if pred != 1: 
            zero_counter +=1 
        else:
            one_counter+=1 
        
    
    print(f"Samples still adversarial: {zero_counter}")
    print(f"adversarial_samples identified: {one_counter}")

make_pred_poisoned(remove_samples_from_asamples)

[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correc

In [22]:
###Model transfer ###
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

In [23]:
### Target Models ####
##  Trained on original data
#   No adversarial samples are added to the training data 

### Decision Tree ####
classifier_dt = DecisionTreeClassifier(random_state=42)
classifier_dt.fit(X_train_fi, y_train_fi)
y_pred_dt = classifier_dt.predict(X_test_fi)
accuracy_dt = accuracy_score(y_test_fi, y_pred_dt) 
print(f"Accuracy of Decision Tree Target Model: {accuracy_score(y_test_fi, y_pred_dt):.2f}")

### XGboost #### 
params = {
    'objective':'binary:logistic',
    'max_depth':4,
    'learning_rate':0.1,
    'n_estimators':100,
    'alpha':10
}

model = XGBClassifier(**params)
model.fit(X_train_fi, y_train_fi)
y_pred_xgb = model.predict(X_test_fi)
accuracy_xgb = accuracy_score(y_test_fi, y_pred_xgb)
print(f'Accuracy of the XGboost Model: {accuracy_xgb * 100:.2f}%') 

### SVM Classifier Kernel=Poly ###
svm_classifier = SVC(kernel='poly', C=1.0, random_state=42,max_iter=5000)
svm_classifier.fit(X_train_fi, y_train_fi)
y_pred_svm = svm_classifier.predict(X_test_fi)
print(f"Accuracy of the SVM Classifier (Kernel=Poly): {accuracy_score(y_test_fi, y_pred_svm):.2f}")

### SVM Classifier Kernel=Linear ###
svm_classifier_linear = SVC(kernel='linear', C=4.0, random_state=42,max_iter=5000)
svm_classifier_linear.fit(X_train_fi, y_train_fi)
y_pred_svm_lin = svm_classifier_linear.predict(X_test_fi)
print(f"Accuracy of the SVM Classifier (Kernel=Linear) : {accuracy_score(y_test_fi, y_pred_svm_lin):.2f}")

### Logistic Regression ###
LR_model = LogisticRegression(solver='saga',max_iter=5000)
LR_model.fit(X_train_fi, y_train_fi)
ypred_LR = LR_model.predict(X_test_fi)
accuracy_lr = accuracy_score(y_test_fi, ypred_LR)
print(f"accuracy of the Logistic Regression Model: {round(accuracy_lr,2)}")

Accuracy of Decision Tree Target Model: 0.97
Accuracy of the XGboost Model: 97.77%
Accuracy of the SVM Classifier (Kernel=Poly): 0.80
Accuracy of the SVM Classifier (Kernel=Linear) : 0.86


c:\Users\Neezy\anaconda3\envs\tf\lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=5000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


accuracy of the Logistic Regression Model: 0.69


In [24]:
def evasion_attack(adversarial_samples,
                   DT_model,
                   XGB_Model, 
                   SVM_poly_model, 
                   SVM_linear_model, 
                   Logistic_reg_model):
    
    predictions = {
        "Decision_Tree":[],
        "XGB":[],
        "SVM_poly": [], 
        "SVM_Linear":[], 
        "Logistic_Regression":[]   
    }
    
    models = {
        "Decision_Tree": DT_model,
        "XGB":XGB_Model,
        "SVM_poly": SVM_poly_model, 
        "SVM_Linear":SVM_linear_model, 
        "Logistic_Regression":Logistic_reg_model
    }
    
    for name, model in models.items():
        preds = model.predict(adversarial_samples)
        predictions[name] = preds.tolist()
            
    
    return predictions

In [25]:
res = evasion_attack(asamples, classifier_dt, model, svm_classifier, svm_classifier_linear, LR_model)
evaded_ids=[]
evaded_samples = []
original_positions = []
for model_name, preds in res.items():
    print(f"\n {model_name}: predictions")
    total = len(preds)
    count_1 = sum(1 for p in preds if p==1)
    count_2 = total-count_1 
    
    model_evaded_positions = []
    model_evaded_samples = []
    
    for i, p in enumerate(preds): 
        if p == 0: 
           xtrain_position = adv_idx[i] 
           original_position = idx_train[xtrain_position]
           model_evaded_positions.append(original_position)
            
    if model_evaded_positions:
        for og_position in model_evaded_positions:
            og_sample = data.iloc[og_position]['Sample Name']
            model_evaded_samples.append(og_sample)
        evaded_samples.extend(model_evaded_samples)
        original_positions.extend(model_evaded_positions)
    else: 
        print(f"\n No samples evaded for {model_name}")
    
    print(f"\n {model_name} evasion rate: {count_2}/{total} samples evaded")
    
    sample_names = [s.rsplit('_', 1)[0] for s in model_evaded_samples]
    print("\n evaded samples: ", Counter(sample_names))      


 Decision_Tree: predictions

 Decision_Tree evasion rate: 25/473 samples evaded

 evaded samples:  Counter({'GandCrab': 6, 'None': 4, 'DarkSide': 2, 'Conti': 1, 'Geodo': 1, 'Azov': 1, 'Snatch': 1, 'Sodinokibi': 1, 'DarkyLock': 1, 'HelloXD': 1, 'Yatron': 1, 'RedLineStealer': 1, 'Hydra': 1, 'Ryuk': 1, 'Pandora': 1, 'WannaCry': 1})

 XGB: predictions

 XGB evasion rate: 1/473 samples evaded

 evaded samples:  Counter({'CryLock': 1})

 SVM_poly: predictions

 No samples evaded for SVM_poly

 SVM_poly evasion rate: 0/473 samples evaded

 evaded samples:  Counter()

 SVM_Linear: predictions

 SVM_Linear evasion rate: 10/473 samples evaded

 evaded samples:  Counter({'None': 4, 'Snatch': 1, 'Hermes': 1, 'Maze': 1, 'Xorist': 1, 'Hydra': 1, 'RagnarLocker': 1})

 Logistic_Regression: predictions

 Logistic_Regression evasion rate: 325/473 samples evaded

 evaded samples:  Counter({'None': 36, 'GandCrab': 32, 'Azov': 19, 'DarkSide': 13, 'Conti': 12, 'RedLineStealer': 10, 'WannaCry': 9, 'Stop': 8

In [26]:
scaler = StandardScaler()
Xtrain_scaled = scaler.fit_transform(X_train_fi)
Xtest_scaled = scaler.transform(X_test_fi)

In [27]:
### Scaling 
### Scaled Target Models ####
##  Trained on original data
#   No adversarial samples are added to the training data 

### Decision Tree ####
classifier_dt_scaled = DecisionTreeClassifier(random_state=42)
classifier_dt_scaled.fit(Xtrain_scaled, y_train_fi)
y_pred_dt_scaled = classifier_dt_scaled.predict(Xtest_scaled)
accuracy_dt_scaled = accuracy_score(y_test_fi, y_pred_dt_scaled) 
print(f"Accuracy of Decision Tree Target Model: {accuracy_dt_scaled:.2f}")

### XGboost #### 
params = {
    'objective':'binary:logistic',
    'max_depth':4,
    'learning_rate':0.1,
    'n_estimators':100,
    'alpha':10
}

model_scaled = XGBClassifier(**params)
model_scaled.fit(Xtrain_scaled, y_train_fi)
y_pred_xgb_scaled = model_scaled.predict(Xtest_scaled)
accuracy_xgb_scaled = accuracy_score(y_test_fi, y_pred_xgb_scaled)
print(f'Accuracy of the XGboost Model: {accuracy_xgb_scaled * 100:.2f}%') 

### SVM Classifier Kernel=Poly ###
svm_classifier_scaled = SVC(kernel='poly', C=1.0, random_state=42)
svm_classifier_scaled.fit(Xtrain_scaled, y_train_fi)
y_pred_SVM_poly_scaled = svm_classifier_scaled.predict(Xtest_scaled)
print(f"Accuracy of the SVM Classifier (Kernel=Poly): {accuracy_score(y_test_fi, y_pred_SVM_poly_scaled):.2f}")

### SVM Classifier Kernel=Linear ###
svm_classifier_linear_scaled = SVC(kernel='linear', C=4.0, random_state=42)
svm_classifier_linear_scaled.fit(Xtrain_scaled, y_train_fi)
y_pred_SVM_linear_scaled = svm_classifier_linear_scaled.predict(Xtest_scaled)
print(f"Accuracy of the SVM Classifier (Kernel=Linear) : {accuracy_score(y_test_fi, y_pred_SVM_linear_scaled):.2f}")

### Logistic Regression ###
LR_model_scaled = LogisticRegression(solver='saga',max_iter=5000)
LR_model_scaled.fit(Xtrain_scaled, y_train_fi)
ypred_lr_scaled = LR_model_scaled.predict(Xtest_scaled)
accuracy_lr_scaled = accuracy_score(y_test_fi, ypred_lr_scaled)
print(f"accuracy of the Logistic Regression Model: {round(accuracy_lr_scaled,2)}")

Accuracy of Decision Tree Target Model: 0.97
Accuracy of the XGboost Model: 97.77%
Accuracy of the SVM Classifier (Kernel=Poly): 0.96
Accuracy of the SVM Classifier (Kernel=Linear) : 0.97
accuracy of the Logistic Regression Model: 0.96


In [28]:
def evasion_attack_scaled(adversarial_samples_scaled,
                   DT_Model_scaled,
                   XGB_Model_scaled, 
                   SVM_poly_model_scaled, 
                   SVM_linear_model_scaled, 
                   Logistic_reg_model_scaled):
    
    predictions = {
        "DT_Model_scaled":[],
        "XGB_scaled":[],
        "SVM_poly_scaled": [], 
        "SVM_Linear_scaled":[], 
        "Logistic_Regression_scaled":[]   
    }
    
    models = {
        "DT_Model_scaled": DT_Model_scaled,
        "XGB_scaled":XGB_Model_scaled,
        "SVM_poly_scaled": SVM_poly_model_scaled, 
        "SVM_Linear_scaled":SVM_linear_model_scaled, 
        "Logistic_Regression_scaled":Logistic_reg_model_scaled
    }
    
    for name, model in models.items():
        preds = model.predict(adversarial_samples_scaled)
        predictions[name] = preds.tolist()
            
    
    return predictions

In [29]:
asamples_scaled = scaler.transform(asamples)

In [30]:

res_scaled = evasion_attack_scaled(asamples_scaled, classifier_dt_scaled, model_scaled, svm_classifier_scaled, svm_classifier_linear_scaled, LR_model_scaled)
evaded_ids=[]
evaded_samples = []
original_positions = []
for model_name, preds in res_scaled.items():
    print(f"\n {model_name}: predictions")
    total = len(preds)
    count_1 = sum(1 for p in preds if p==1)
    count_2 = total-count_1 
    
    model_evaded_positions = []
    model_evaded_samples = []
    
    for i, p in enumerate(preds): 
        if p == 0: 
           xtrain_position = adv_idx[i] 
           original_position = idx_train[xtrain_position]
           model_evaded_positions.append(original_position)
            
    if model_evaded_positions:
        for og_position in model_evaded_positions:
            og_sample = data.iloc[og_position]['Sample Name']
            model_evaded_samples.append(og_sample)
        evaded_samples.extend(model_evaded_samples)
        original_positions.extend(model_evaded_positions)
    else: 
        print(f"\n No samples evaded for {model_name}")
    
    print(f"\n {model_name} evasion rate: {count_2}/{total} samples evaded")
    
    sample_names = [s.rsplit('_', 1)[0] for s in model_evaded_samples]
    print("\n evaded samples: ", Counter(sample_names)) 


 DT_Model_scaled: predictions

 DT_Model_scaled evasion rate: 25/473 samples evaded

 evaded samples:  Counter({'GandCrab': 6, 'None': 4, 'DarkSide': 2, 'Conti': 1, 'Geodo': 1, 'Azov': 1, 'Snatch': 1, 'Sodinokibi': 1, 'DarkyLock': 1, 'HelloXD': 1, 'Yatron': 1, 'RedLineStealer': 1, 'Hydra': 1, 'Ryuk': 1, 'Pandora': 1, 'WannaCry': 1})

 XGB_scaled: predictions

 XGB_scaled evasion rate: 1/473 samples evaded

 evaded samples:  Counter({'CryLock': 1})

 SVM_poly_scaled: predictions

 SVM_poly_scaled evasion rate: 9/473 samples evaded

 evaded samples:  Counter({'Babuk': 6, 'DarkyLock': 1, 'Maze': 1, 'Xorist': 1})

 SVM_Linear_scaled: predictions

 SVM_Linear_scaled evasion rate: 1/473 samples evaded

 evaded samples:  Counter({'CryLock': 1})

 Logistic_Regression_scaled: predictions

 Logistic_Regression_scaled evasion rate: 389/473 samples evaded

 evaded samples:  Counter({'GandCrab': 31, 'None': 30, 'UpdateInformation.html': 16, 'UpdateRecords.html': 15, 'DarkSide': 13, 'RedLineStealer